# 11 — Experimento de no respuesta (bloque C, Fase D)

¿Cambia el modelo según cómo se trata la no respuesta de ingresos en el **train**? Contra la referencia
`imputacion_excluye` (bloque A, ya en `data/07_predicciones/`, no se reentrena) se entrenan tres niveles con
LightGBM, universo 2003–2025, h ∈ {1, 4}, trayectoria simétrica, entrada y permanencia, con los hiperparámetros del
bloque A (`modelos.HIPERPARAMETROS_GB`):

- `imputacion_excluye_fraccion_max_0.5`: conserva los pares con fracción imputada ≤ 0,5 en las dos puntas;
- `imputacion_excluye_reponderado`: pares de `excluye`, con el peso de los imputados repartido en su celda
  aglomerado × trimestre de origen;
- `imputacion_excluye_reponderado_propension`: pares de `excluye`, con peso `1/max(p̂, 0,05)` recortado al p99 por
  era, donde `p̂` es la propensión de par válido del 05f, reestimada dentro del train de cada fold.

Los pesos van solo en el entrenamiento. Hay cuatro lecturas del test, siempre sobre el conjunto limpio y además
sobre los imputados:

| Lectura | Peso | A quién representa |
|---|---|---|
| sin ponderar | ninguno | declarantes; criterio principal |
| `peso_normalizado_trimestre` | ponderador de la encuesta | población expandida |
| `ponderado_propension` | propensión del train del fold, aplicada a las X en `t` del test | población completa, declarantes y no declarantes |
| imputados | ninguno | folds históricos |

Los niveles reponderados apuntan a la población completa, así que su comparación principal contra `excluye` se lee
en la lectura ponderada por propensión. Que pierdan en la sin ponderar no implica que sean peores.

Además se corre la sensibilidad por aglomerados de baja respuesta y la caracterización de esos aglomerados (como
hipótesis). Las reglas están en `.claude/decisiones.md`. **El holdout no se abre**: todo sale de pares con destino
≤ 2022T4.

La variable de entorno `BLOQUE_C_FOLDS` (JSON `{"h": [años]}`) limita los folds para la corrida corta de medición
de costo. Sin ella corren todos. Cada fold ya escrito y verificado se saltea, salvo `REHACER`.

In [ ]:
import json
import os
import sys
import time
from pathlib import Path


def _raiz_repo(desde: Path | None = None) -> Path:
    actual = (desde or Path.cwd()).resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pytest.ini").exists():
            return candidato
    raise FileNotFoundError(f"no se encontró la raíz del repo subiendo desde {actual}")


REPO = _raiz_repo()
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from joblib import Parallel, delayed
from scipy.stats import spearmanr

import constantes as cte
import modelos as mo
import no_respuesta_cotas as nc
import no_respuesta_modelo as nm
import panel
import pipeline_meta as pm
import universo_entrenamiento as ue
import validacion as va
from armonizacion import trimestres_publicados
from features import CLAVE_PAR

REHACER: bool = False
N_BOOT: int = 200
UMBRAL_BAJA_RESPUESTA = 0.70
N_COMPARABLES = 3
AGLOMERADOS_CARACTERIZAR = (2, 3, 34)
FOLDS_LIMITADOS = json.loads(os.environ.get("BLOQUE_C_FOLDS", "null"))
NOTEBOOK = "11_experimento_no_respuesta"
HORIZONTES = [1, 4]
OBJETIVOS = cte.OBJETIVOS_MODELADO
PROPENSION = "imputacion_excluye_reponderado_propension"
PERFIL = ["era", "situacion_laboral_t", "condicion_actividad_jefe_t", "nivel_ed_jefe_bucket_t", "tipologia_perceptores_t"]
EXTRA = ("tasa_pobreza_aglo_t", "gini_ipcf_aglo_t", *PERFIL[1:])

FEATURES = REPO / "data" / "06_features"
TRAYECTORIA = REPO / "data" / "05_trayectoria_distancia"
ARMONIZADO = REPO / "data" / "01_armonizado"
DISTANCIA = REPO / "data" / "02b_distancia_linea_pobreza" / "hogar"
PREDICCIONES = REPO / "data" / "07_predicciones"
META = REPO / "data" / "meta"
PREFIJO = "experimento_no_respuesta"
COMMIT_GIT = pm.obtener_commit_git(REPO)
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 40)
N_PROCESOS = 7
# Los procesos del bootstrap importan `modelos` por nombre: necesitan `src` en su ruta.
os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO / "src"), os.environ.get("PYTHONPATH", "")])
TIEMPOS: dict[str, float] = {}
t0_proceso = time.time()


def medir(etapa: str, t0: float) -> None:
    TIEMPOS[etapa] = TIEMPOS.get(etapa, 0.0) + time.time() - t0
    print(f"[{etapa}] {time.time() - t0:.0f}s (acumulado {time.time() - t0_proceso:.0f}s)", flush=True)


def guardar(tabla: pd.DataFrame, nombre: str) -> None:
    tabla.to_csv(META / f"{PREFIJO}_{nombre}.csv", index=False)


def evaluar_tareas(tareas: list[tuple[dict, int]]) -> pd.DataFrame:
    preparados = [(mo.preparar_evaluacion_fold(**argumentos), n_boot) for argumentos, n_boot in tareas]
    tablas = Parallel(n_jobs=N_PROCESOS)(delayed(mo.evaluar_preparado)(p, n) for p, n in preparados if p is not None)
    tablas = [t for t in tablas if not t.empty]
    return pd.concat(tablas, ignore_index=True) if tablas else pd.DataFrame()

## Consolidación del registro de corridas

Las corridas del 05e y el 05f se registraron en `data/meta/corridas_no_respuesta.csv`. Esta celda agrega a
`corridas.csv` las filas que todavía no están (clave `notebook` + `fecha_hora`), tal como figuran en el original.
Es idempotente y no borra nada: `corridas_no_respuesta.csv` queda en su lugar.

In [ ]:
corridas = pd.read_csv(META / "corridas.csv", dtype=str)
diagnostico = pd.read_csv(META / "corridas_no_respuesta.csv", dtype=str)
assert list(diagnostico.columns) == list(corridas.columns)
clave_corrida = ["notebook", "fecha_hora"]
faltan = diagnostico.merge(corridas[clave_corrida], on=clave_corrida, how="left", indicator=True)
faltan = faltan[faltan["_merge"] == "left_only"].drop(columns="_merge")
if len(faltan):
    faltan.to_csv(META / "corridas.csv", mode="a", header=False, index=False)
print(f"filas agregadas a corridas.csv: {len(faltan)} (de {len(diagnostico)} del diagnóstico)")

## Pares de train, folds y referencia

Igual que el 10: se cargan con `imputacion_conserva` (el superconjunto) y cada nivel filtra su train. Se verifica que
la referencia (`excluye`, decil y GB) esté escrita con el `configuracion_id` del bloque A.

Los folds donde la referencia no existe no se corren: son h=1 2005 y h=4 2006–2007, omitidos en el bloque A por
validación interna no utilizable (train demasiado chico). Sin referencia, no hay contra qué comparar.

In [ ]:
t0 = time.time()
pares, folds, anios = {}, {}, {}
for h in HORIZONTES:
    p = va.cargar_pares_modelado(FEATURES, TRAYECTORIA, h, mo.columnas_extra_carga(h) + EXTRA,
                                 criterio_imputacion="imputacion_conserva")
    pares[h] = p[ue.en_universo(p, "universo_2003_2025")].reset_index(drop=True)
    pares[h]["aglomerado"] = pares[h]["AGLOMERADO_t"].astype(int)
    va.exigir_sin_holdout(pares[h])
    folds[h] = va.generar_folds(pares[h][va.mascara_train(pares[h])], h)
    todos = folds[h]["anio_test"].tolist()
    anios[h] = todos if FOLDS_LIMITADOS is None else [a for a in FOLDS_LIMITADOS.get(str(h), []) if a in todos]
    print(f"h={h}: {len(pares[h]):,} pares, folds a correr {anios[h]}")
HISTORICOS = {h: set(folds[h].loc[folds[h]["era_test"] == "historica", "anio_test"]) for h in HORIZONTES}

ids_referencia, sin_referencia = [], {h: set() for h in HORIZONTES}
for h in HORIZONTES:
    for objetivo in OBJETIVOS:
        for configuracion in mo.configuraciones_referencia(objetivo, h):
            directorio = mo.directorio_configuracion(PREDICCIONES, configuracion, mo.id_configuracion(configuracion))
            assert (directorio / "manifest.json").exists(), f"falta la referencia {configuracion.nombre} {objetivo} h={h}"
            faltantes = [a for a in anios[h] if not mo.fold_completo(directorio, a)]
            sin_referencia[h] |= set(faltantes)
            ids_referencia.append({"objetivo_modelado": objetivo, "h": h, "configuracion": configuracion.nombre,
                                   "configuracion_id": mo.id_configuracion(configuracion), "folds_sin_escribir": faltantes})
for h in HORIZONTES:
    anios[h] = [a for a in anios[h] if a not in sin_referencia[h]]
    print(f"h={h}: folds sin referencia (omitidos en el bloque A) {sorted(sin_referencia[h])}; se corren {anios[h]}")
display(pd.DataFrame(ids_referencia))
medir("carga_pares", t0)

## Hogares de train: variables sin ingreso y declaración

Una fila por hogar de los trimestres ≤ 2022T4, con las variables de `nm.variables_hogar` (sin ningún monto) y la
marca de declaración de `nc.clases_cota_*`:

- era regular: no declara si `ingreso_no_declarado` (`PONDIH == 0`);
- era histórica: no declara si tiene alguna marca de imputación en `IDIMPH`.

Es la misma construcción del 05f.

In [ ]:
t0 = time.time()
TRIMESTRES_TRAIN = [t for t in trimestres_publicados() if t <= va.ULTIMO_DESTINO_TRAIN]


def era_de(anio: int, trimestre: int) -> str:
    return "historica" if (anio, trimestre) <= (2015, 2) else "regional_2016"


partes_variables, partes_clases = [], []
for anio, trimestre in TRIMESTRES_TRAIN:
    partes_variables.append(nm.variables_hogar(anio, trimestre, ARMONIZADO).assign(era=era_de(anio, trimestre)))
    funcion = nc.clases_cota_historica if era_de(anio, trimestre) == "historica" else nc.clases_cota_regular
    partes_clases.append(funcion(anio, trimestre, ARMONIZADO, DISTANCIA)[0][[*nm.CLAVE_HOGAR_TRIMESTRE, "con_no_respuesta",
                                                                            "clase"]])
variables = pd.concat(partes_variables, ignore_index=True)
clases = pd.concat(partes_clases, ignore_index=True)
del partes_variables, partes_clases
COLUMNAS_PROPENSION = nm.columnas_propension(variables)
hogares = variables.merge(clases, on=nm.CLAVE_HOGAR_TRIMESTRE, how="inner", validate="one_to_one")
assert len(hogares) == len(clases)
hogares["declara"] = (~hogares["con_no_respuesta"].astype(bool)).astype(int)
hogares["aglomerado"] = hogares["AGLOMERADO"].astype(int)
print(f"{len(hogares):,} hogares de train, {len(COLUMNAS_PROPENSION)} variables de propensión")
display(hogares.groupby("era")["declara"].agg(["size", "mean"]).round(4))
medir("hogares", t0)

## Aglomerados de baja respuesta

La regla quedó fijada en `decisiones.md` antes de esta corrida. Se calcula la tasa de declaración por aglomerado
sobre los hogares de train, con las dos eras juntas y sin ponderar. Baja respuesta = tasa < 0,70. Nombres según el
codebook INDEC.

También se informa qué % de los pares válidos de train (`excluye`) caen en esos aglomerados. Los aglomerados 2, 3 y
34 se caracterizan siempre, pasen o no el umbral.

In [ ]:
tasa_aglomerado = hogares.groupby("aglomerado")["declara"].agg(n_hogares="size", tasa_declaracion="mean")
tasa_era = hogares.pivot_table(index="aglomerado", columns="era", values="declara", aggfunc="mean")
tasa_era.columns = [f"tasa_declaracion_{c}" for c in tasa_era.columns]
tabla_respuesta = tasa_aglomerado.join(tasa_era).reset_index()
tabla_respuesta.insert(1, "nombre_codebook", tabla_respuesta["aglomerado"].map(nm.NOMBRES_AGLOMERADO))
assert tabla_respuesta["nombre_codebook"].notna().all(), "aglomerado sin nombre en el codebook"
tabla_respuesta["baja_respuesta"] = tabla_respuesta["tasa_declaracion"] < UMBRAL_BAJA_RESPUESTA
BAJA = tuple(int(a) for a in tabla_respuesta.loc[tabla_respuesta["baja_respuesta"], "aglomerado"])
CARACTERIZAR = tuple(sorted(set(AGLOMERADOS_CARACTERIZAR) | set(BAJA)))
for h in HORIZONTES:
    validos = pares[h][va.mascara_train(pares[h])]
    share = validos.groupby("aglomerado").size() / len(validos)
    tabla_respuesta[f"pct_pares_validos_train_h{h}"] = 100 * tabla_respuesta["aglomerado"].map(share).fillna(0.0)
guardar(tabla_respuesta, "baja_respuesta")
print("baja respuesta:", {a: nm.NOMBRES_AGLOMERADO[a] for a in BAJA})
print("caracterización:", CARACTERIZAR)
for h in HORIZONTES:
    print(f"h={h}: % de pares válidos de train en baja respuesta = "
          f"{tabla_respuesta.loc[tabla_respuesta['baja_respuesta'], f'pct_pares_validos_train_h{h}'].sum():.2f}")
display(tabla_respuesta.sort_values("tasa_declaracion").round(4))

## Pérdida de pares por no respuesta, por aglomerado, era y h

Pares de train con identidad confirmada, y de ellos los que se pierden por ingreso: no son válidos y alguna punta no
declara o está imputada.

El % observado se contrasta con el esperado si la no respuesta fuera independiente entre visitas, `1 − d²`, donde
`d` es la tasa de declaración de hogares del aglomerado en esa era. Si el observado queda por debajo del esperado,
la no respuesta persiste en el mismo hogar: el que no declara en `t` tiende a no declarar en `t+h`, y se pierden
menos pares de lo que daría la independencia.

In [ ]:
t0 = time.time()
pares05 = {}
for h in HORIZONTES:
    crudo = nm.leer_pares_05(TRAYECTORIA, h)
    crudo = crudo[panel.particion_train_holdout(crudo, va.HOLDOUT_DESDE) == "train"].reset_index(drop=True)
    pares05[h] = nm.marcar_perdidos_por_ingreso(crudo, clases)
    pares05[h]["aglomerado"] = pares05[h]["AGLOMERADO_t"].astype(int)

d_era = hogares.groupby(["aglomerado", "era"])["declara"].mean().rename("d")
filas = []
for h, p in pares05.items():
    identidad = p[p["identidad_confirmada"].astype("boolean").fillna(False).astype(bool)]
    agregado = identidad.groupby(["aglomerado", "era_t"]).agg(
        n_identidad=("perdido_por_ingreso", "size"), n_perdidos_por_ingreso=("perdido_por_ingreso", "sum")).reset_index()
    filas.append(agregado.assign(h=h))
perdida = pd.concat(filas, ignore_index=True).rename(columns={"era_t": "era"})
perdida = perdida.merge(d_era.reset_index(), on=["aglomerado", "era"], how="left")
perdida["pct_perdidos"] = 100 * perdida["n_perdidos_por_ingreso"] / perdida["n_identidad"]
perdida["pct_esperado_independencia"] = 100 * (1 - perdida["d"] ** 2)
perdida["diferencia_pp"] = perdida["pct_perdidos"] - perdida["pct_esperado_independencia"]
perdida["cociente"] = perdida["pct_perdidos"] / perdida["pct_esperado_independencia"]
perdida.insert(1, "nombre_codebook", perdida["aglomerado"].map(nm.NOMBRES_AGLOMERADO))
perdida["baja_respuesta"] = perdida["aglomerado"].isin(BAJA)
guardar(perdida, "perdida_pares_aglomerado")
display(perdida.pivot_table(index=["aglomerado", "nombre_codebook"], columns=["era", "h"],
                            values=["pct_perdidos", "pct_esperado_independencia", "cociente"]).round(2))
display(perdida.groupby(["baja_respuesta", "era", "h"])[["pct_perdidos", "pct_esperado_independencia", "cociente"]].median().round(3))
medir("perdida_pares", t0)

## Propensión de par válido por fold

La población es la del 05f §2c: pares de train con identidad confirmada que hoy son válidos o que se pierden solo
por ingreso, con las X del hogar en `t`. En cada fold, con el train del fold y nada más:

- **pesos de train**: un LightGBM por era, con predicción fuera de fold en 5 grupos por vivienda dentro del train;
  peso `1/max(p̂, 0,05)` de los válidos, recortado al p99 de esa era en ese train;
- **pesos de test**: un modelo por era reajustado sobre todo el train del fold, aplicado a las X en `t` de los pares
  limpios del test, con el **mismo corte p99 del train**. Esas X no dependen del ingreso, así que no hay fuga.

`nm.pesos_propension_test` corta si algún par del test está entre las filas de ajuste. Una era del test sin modelo
en el train (el fold 2016, por el salto de era) queda en NA en la lectura ponderada.

Los pesos se guardan junto a la configuración de trayectoria del nivel (`propension_{train,test}.parquet` por fold),
para reanudar sin reestimar.

In [ ]:
t0 = time.time()
poblacion = {}
for h, p in pares05.items():
    sub = p[p["valido_actual"] | p["perdido_por_ingreso"]]
    poblacion[h] = nm.unir_variables_t(sub, variables).reset_index(drop=True)
    sin_x = int(poblacion[h]["aglomerado_cat"].isna().sum())
    print(f"h={h}: población de propensión {len(poblacion[h]):,} pares ({sin_x} sin fila de hogar en t)")
del variables


def directorio_nivel(objetivo: str, h: int, criterio: str) -> Path:
    configuracion = mo.configuracion_gb(objetivo, h, criterio)
    return mo.directorio_configuracion(PREDICCIONES, configuracion, mo.id_configuracion(configuracion))


def _serie(ruta: Path) -> pd.Series:
    return pd.read_parquet(ruta).set_index(CLAVE_PAR)["peso"]


pesos_train = {h: {} for h in HORIZONTES}
pesos_test = {h: {} for h in HORIZONTES}
metricas_propension = []
for h in HORIZONTES:
    for anio in anios[h]:
        cache = directorio_nivel("trayectoria", h, PROPENSION) / f"fold={anio}"
        rutas = {nombre: cache / f"propension_{nombre}.parquet" for nombre in ("train", "test")}
        if not REHACER and all(r.exists() for r in rutas.values()) and (cache / "propension_metricas.csv").exists():
            pesos_train[h][anio], pesos_test[h][anio] = _serie(rutas["train"]), _serie(rutas["test"])
            metricas_propension.append(pd.read_csv(cache / "propension_metricas.csv"))
            continue
        t_fold = time.time()
        rol = va.asignar_fold(poblacion[h], anio)
        ajuste = nm.ajustar_propension_fold(poblacion[h][rol == "train"], COLUMNAS_PROPENSION, num_threads=mo.HILOS)
        test = poblacion[h][(rol == "test") & poblacion[h]["valido_actual"]]
        pesos_train[h][anio], pesos_test[h][anio] = ajuste["pesos"], nm.pesos_propension_test(ajuste, test)
        cache.mkdir(parents=True, exist_ok=True)
        for nombre, serie in (("train", pesos_train[h][anio]), ("test", pesos_test[h][anio])):
            serie.rename("peso").reset_index().to_parquet(rutas[nombre], index=False)
        tabla = ajuste["metricas"].assign(h=h, anio_test=anio, corte_p99=lambda d: d["era"].map(ajuste["cortes"]),
                                          n_test=len(test), n_test_con_peso=int(pesos_test[h][anio].notna().sum()),
                                          segundos=time.time() - t_fold)
        tabla.to_csv(cache / "propension_metricas.csv", index=False)
        metricas_propension.append(tabla)
        print(f"h={h} fold {anio}: {time.time() - t_fold:.0f}s", flush=True)
        del ajuste
metricas_propension = pd.concat(metricas_propension, ignore_index=True) if metricas_propension else pd.DataFrame()
guardar(metricas_propension, "propension_fold")
display(metricas_propension[["h", "anio_test", "era", "n", "auc", "tasa_observada", "media_predicha", "corte_p99",
                             "peso_n_efectivo_relativo", "n_test", "n_test_con_peso"]].round(3))
medir("propension_fold", t0)

## Entrenamiento de los tres niveles y del modelo de contaminación

Por (objetivo, h) se entrenan los tres niveles con GB y los hiperparámetros del bloque A. Cada uno tiene su early
stopping, y la validación interna queda sin ponderar.

Para la sensibilidad (a) se entrena además el modelo de contaminación: `excluye` sin los aglomerados de baja
respuesta en el train, con el test entero.

En cada fold se guardan el booster (`modelo.txt`, para el SHAP del bloque E) y, en los niveles con peso,
`pesos_train.parquet`.

In [ ]:
t0 = time.time()
ajustes = []
for h in HORIZONTES:
    for objetivo in OBJETIVOS:
        configuraciones = [mo.configuracion_gb(objetivo, h, criterio) for criterio in mo.CRITERIOS_BLOQUE_C]
        if BAJA:
            configuraciones.append(mo.configuracion_gb(objetivo, h, cte.CRITERIO_IMPUTACION_PRINCIPAL, BAJA))
        for configuracion in configuraciones:
            externos = pesos_train[h] if configuracion.criterio_imputacion == PROPENSION else None
            registro = mo.correr_configuracion(pares[h], anios[h], configuracion, PREDICCIONES, REPO, NOTEBOOK, REHACER,
                                               pesos_externos=externos, guardar_modelo=True)
            ajustes.append(registro.assign(objetivo_modelado=objetivo, h=h, criterio_imputacion=configuracion.criterio_imputacion))
            print(f"h={h} {objetivo} {configuracion.nombre}: {time.time() - t0:.0f}s", flush=True)
ajustes = pd.concat(ajustes, ignore_index=True)
for columna in ("error", "omitido"):
    if columna not in ajustes:
        ajustes[columna] = pd.NA
errores = ajustes[ajustes["error"].notna()]
print(f"folds con error: {len(errores)}; omitidos: {int(ajustes['omitido'].notna().sum())}")
display(errores[["objetivo_modelado", "h", "configuracion", "anio_test", "error"]])
ajustes["train_chico"] = ajustes["n_final"] < mo.UMBRAL_TRAIN_CHICO
guardar(ajustes, "ajuste_fold")
medir("entrenamiento", t0)

### Verificación de las predicciones fuera de muestra

In [ ]:
CONFIGURACIONES = {}
for h in HORIZONTES:
    for objetivo in OBJETIVOS:
        referencia = mo.configuraciones_referencia(objetivo, h)
        niveles = [mo.configuracion_gb(objetivo, h, criterio) for criterio in mo.CRITERIOS_BLOQUE_C]
        contaminacion = [mo.configuracion_gb(objetivo, h, cte.CRITERIO_IMPUTACION_PRINCIPAL, BAJA)] if BAJA else []
        CONFIGURACIONES[(objetivo, h)] = {"referencia": referencia, "niveles": niveles, "contaminacion": contaminacion}
escritas = 0
for grupos in CONFIGURACIONES.values():
    for configuracion in grupos["niveles"] + grupos["contaminacion"]:
        escritas += len(mo.leer_predicciones(PREDICCIONES, configuracion))
print(f"verificador OK: {escritas:,} predicciones nuevas")

## Diferencias por fold: cada nivel contra `excluye`

En cada fold y lectura hay un único bootstrap por vivienda, compartido por todas las configuraciones, así que las
diferencias nivel − excluye salen pareadas.

Lecturas:

- limpio sin ponderar, con IC;
- limpio `peso_normalizado_trimestre`, sin IC, como en el 10;
- limpio `ponderado_propension`, con IC;
- imputados en los folds históricos, con IC.

La base del skill es el decil de `excluye`.

In [ ]:
t0 = time.time()


def lecturas(h: int, anio: int) -> list[tuple[str, str, int]]:
    salida = [("limpio", "sin_ponderar", N_BOOT), ("limpio", "ponderado_trimestre", 0),
              ("limpio", cte.PONDERACION_PROPENSION, N_BOOT)]
    if anio in HISTORICOS[h]:
        salida.append(("imputados", "sin_ponderar", N_BOOT))
    return salida


evaluaciones = []
for (objetivo, h), grupos in CONFIGURACIONES.items():
    lista = grupos["referencia"] + grupos["niveles"]
    tareas = []
    for anio in anios[h]:
        for conjunto, ponderacion, n_boot in lecturas(h, anio):
            peso = pesos_test[h][anio] if ponderacion == cte.PONDERACION_PROPENSION else None
            tareas.append(({"pares": pares[h], "raiz": PREDICCIONES, "configuraciones": lista, "anio_test": int(anio),
                            "conjunto": conjunto, "ponderacion": ponderacion, "peso_test": peso}, n_boot))
    evaluaciones.append(evaluar_tareas(tareas))
    print(f"h={h} {objetivo}: {time.time() - t0:.0f}s", flush=True)
metricas = pd.concat([e for e in evaluaciones if not e.empty], ignore_index=True)
marcas = pd.concat([f.assign(h=h) for h, f in folds.items()])[["h", "anio_test", "contexto_fold", "periodo_intervenido"]]
metricas = metricas.merge(marcas, on=["h", "anio_test"], how="left")
guardar(metricas, "metricas_fold")
diferencias = metricas[metricas["configuracion"].str.endswith("_menos_excluye")].copy()
diferencias["ic_excluye_0"] = (diferencias["ic95_low"] > 0) | (diferencias["ic95_high"] < 0)
guardar(diferencias, "diferencias_fold")
medir("evaluacion", t0)

### Resumen entre folds, con discriminación y calibración por separado

Por (objetivo, h, nivel, lectura): mediana de la diferencia nivel − excluye entre folds, y cuántos folds tienen un
IC95% que excluye 0 en cada sentido.

- Clasificación: discriminación = AUC-ROC, AUC-PR relativa, log loss, Brier. Calibración en grande = media predicha −
  tasa, intercepto, pendiente, ECE.
- Trayectoria: MAE, RMSE, skill contra el decil.

Las dos columnas principales van lado a lado: sin ponderar (declarantes) y ponderada por propensión (población
completa).

In [ ]:
DISCRIMINACION = {"clasificacion": ["auc_roc", "auc_pr_relativa", "log_loss", "brier"],
                  "trayectoria": ["rmse", "mae", "skill_vs_base"]}
CALIBRACION = ["diferencia_media_predicha", "intercepto_calibracion", "pendiente_calibracion", "ece"]
LECTURA = {("limpio", "sin_ponderar"): "sin_ponderar", ("limpio", "ponderado_trimestre"): "ponderado_trimestre",
           ("limpio", cte.PONDERACION_PROPENSION): "ponderado_propension", ("imputados", "sin_ponderar"): "imputados"}


def resumir(tabla: pd.DataFrame) -> pd.DataFrame:
    tabla = tabla.assign(lectura=[LECTURA[(c, p)] for c, p in zip(tabla["conjunto_evaluacion"], tabla["ponderacion"])],
                         nivel=tabla["configuracion"].str.removeprefix("gradient_boosting|").str.removesuffix("_menos_excluye"))
    con_ic = tabla["ic95_low"].notna()
    tabla["favor_positivo"] = con_ic & (tabla["ic95_low"] > 0)
    tabla["favor_negativo"] = con_ic & (tabla["ic95_high"] < 0)
    return tabla.groupby(["objetivo_modelado", "h", "nivel", "metrica", "lectura"]).agg(
        mediana=("valor", "median"), minimo=("valor", "min"), maximo=("valor", "max"), n_folds=("valor", "size"),
        ic_mayor_0=("favor_positivo", "sum"), ic_menor_0=("favor_negativo", "sum")).reset_index()


resumen = resumir(diferencias[diferencias["configuracion"].str.startswith("gradient_boosting|")])
guardar(resumen, "resumen_diferencias")
for bloque, metricas_bloque in (("discriminación", None), ("calibración", CALIBRACION)):
    for objetivo in OBJETIVOS:
        tipo = "trayectoria" if objetivo == "trayectoria" else "clasificacion"
        lista = metricas_bloque if metricas_bloque is not None else DISCRIMINACION[tipo]
        sub = resumen[(resumen["objetivo_modelado"] == objetivo) & resumen["metrica"].isin(lista)
                      & resumen["lectura"].isin(["sin_ponderar", "ponderado_propension"])]
        if sub.empty:
            continue
        print(f"{bloque} — {objetivo}")
        display(sub.pivot_table(index=["h", "nivel", "metrica"], columns="lectura",
                                values=["mediana", "ic_mayor_0", "ic_menor_0"], aggfunc="first").round(4))

## Anomalía de `conserva` en permanencia

Con `conserva`, el GB de permanencia subpredice: media predicha − tasa de −0,023 (h=1) y −0,042 (h=4) en el bloque A.
¿Algún nivel nuevo la reproduce? Por fold, media predicha − tasa y pendiente de calibración del GB `excluye`,
`conserva` (sus predicciones del bloque A, sin reentrenar) y los tres niveles, en las lecturas sin ponderar y
ponderada por propensión. Sin bootstrap.

In [ ]:
t0 = time.time()
filas = []
for h in HORIZONTES:
    grupos = CONFIGURACIONES[("permanencia", h)]
    conserva = [c for c in mo.configuraciones_bloque_a("permanencia", "simetrica", h)
                if c.tecnica == "gradient_boosting" and c.criterio_imputacion == "imputacion_conserva"]
    lista = grupos["referencia"] + conserva + grupos["niveles"]
    for anio in anios[h]:
        for ponderacion in ("sin_ponderar", cte.PONDERACION_PROPENSION):
            peso = pesos_test[h][anio] if ponderacion == cte.PONDERACION_PROPENSION else None
            tabla = mo.evaluar_configuraciones_fold(pares[h], PREDICCIONES, lista, int(anio), "limpio", ponderacion, 0,
                                                    peso_test=peso)
            if not tabla.empty:
                filas.append(tabla[tabla["metrica"].isin(["diferencia_media_predicha", "pendiente_calibracion"])
                                   & ~tabla["configuracion"].str.contains("menos")])
anomalia = pd.concat(filas, ignore_index=True) if filas else pd.DataFrame()
guardar(anomalia, "anomalia_permanencia")
if not anomalia.empty:
    gb = anomalia[anomalia["configuracion"].str.startswith("gradient_boosting")]
    display(gb.pivot_table(index=["h", "metrica", "configuracion"], columns="ponderacion", values="valor",
                           aggfunc="median").round(4))
    display(gb[gb["metrica"] == "diferencia_media_predicha"].pivot_table(
        index=["h", "anio_test"], columns=["ponderacion", "configuracion"], values="valor").round(3))
medir("anomalia", t0)

## Perfiles: el 10% de mayor riesgo

Es el criterio preregistrado. El de importancia (|SHAP|) queda para el bloque E, con los `modelo.txt`.

Por fold, sobre el conjunto limpio, se compara el 10% de mayor riesgo de cada nivel con el de `excluye`. Mayor
riesgo = mayor probabilidad en entrada y permanencia, y variación predicha más negativa en trayectoria.

- Jaccard entre los dos conjuntos, con umbral 0,7.
- Composición de cada 10% por `era` y las categóricas del perfil. Se reporta la mayor diferencia en pp entre
  categorías, con umbral de 5pp.

In [ ]:
t0 = time.time()


def predicciones_limpias(configuracion: mo.Configuracion, h: int) -> pd.DataFrame:
    tabla = mo.leer_predicciones(PREDICCIONES, configuracion)
    tabla = tabla[(tabla["conjunto_evaluacion"] == "limpio") & tabla["fold"].astype(int).isin(anios[h])]
    return tabla[[*CLAVE_PAR, "fold", "prediccion", "y_observado"]]


filas_perfil = []
for (objetivo, h), grupos in CONFIGURACIONES.items():
    gb_excluye = [c for c in grupos["referencia"] if c.tecnica == "gradient_boosting"][0]
    referencia = predicciones_limpias(gb_excluye, h)
    perfil = pares[h][[*CLAVE_PAR, *PERFIL]]
    for nivel in grupos["niveles"]:
        comparado = referencia.merge(predicciones_limpias(nivel, h), on=[*CLAVE_PAR, "fold"], suffixes=("_ref", "_nivel"))
        comparado = comparado.merge(perfil, on=CLAVE_PAR, how="left", validate="many_to_one")
        signo = -1.0 if objetivo == "trayectoria" else 1.0
        for anio, g in comparado.groupby("fold"):
            alto_ref = signo * g["prediccion_ref"] >= np.quantile(signo * g["prediccion_ref"], 0.9)
            alto_nivel = signo * g["prediccion_nivel"] >= np.quantile(signo * g["prediccion_nivel"], 0.9)
            jaccard = (alto_ref & alto_nivel).sum() / (alto_ref | alto_nivel).sum()
            fila = {"objetivo_modelado": objetivo, "h": h, "nivel": nivel.criterio_imputacion, "anio_test": int(anio),
                    "n": len(g), "jaccard_top10": float(jaccard)}
            for variable in PERFIL:
                valores = g[variable].astype("object").fillna("NA").astype(str)
                composicion = pd.concat([valores[alto_ref].value_counts(normalize=True).rename("ref"),
                                         valores[alto_nivel].value_counts(normalize=True).rename("nivel")], axis=1).fillna(0)
                fila[f"max_dif_pp_{variable}"] = float(100 * (composicion["nivel"] - composicion["ref"]).abs().max())
            filas_perfil.append(fila)
perfiles = pd.DataFrame(filas_perfil)
if not perfiles.empty:
    columnas_dif = [c for c in perfiles.columns if c.startswith("max_dif_pp_")]
    perfiles["max_dif_pp"] = perfiles[columnas_dif].max(axis=1)
    perfiles["coincide"] = (perfiles["jaccard_top10"] >= 0.7) & (perfiles["max_dif_pp"] <= 5)
    guardar(perfiles, "perfiles_top10")
    display(perfiles.groupby(["objetivo_modelado", "h", "nivel"]).agg(
        jaccard_mediano=("jaccard_top10", "median"), jaccard_min=("jaccard_top10", "min"),
        max_dif_pp_mediana=("max_dif_pp", "median"), folds_coinciden=("coincide", "sum"), n_folds=("coincide", "size")).round(3))
medir("perfiles", t0)

## Sensibilidad por aglomerados de baja respuesta

Ningún aglomerado se excluye del análisis principal. Esta sección mide cuánto dependen los resultados de los
aglomerados de baja respuesta.

### Deciles: cortes y métricas sin esos aglomerados

En cada fold, los cortes de `ratio_linea_pobreza_t` del train (población del objetivo, `excluye`) con y sin los
aglomerados de baja respuesta. Después, el decil reajustado sin ellos contra el decil completo, los dos evaluados
sobre el resto (test limpio sin esos aglomerados).

In [ ]:
t0 = time.time()


def metricas_punto(objetivo: str, y: pd.Series, prediccion: pd.Series, media: float) -> dict:
    if va.es_clasificacion(objetivo):
        return va.metricas_clasificacion(y, prediccion, None, media)
    return va.metricas_regresion(y, prediccion, media)


filas_cortes, filas_decil = [], []
if BAJA:
    for h in HORIZONTES:
        for objetivo in OBJETIVOS:
            for anio in anios[h]:
                train, test = mo.datos_fold(pares[h], anio, objetivo, "simetrica", cte.CRITERIO_IMPUTACION_PRINCIPAL)
                baja_train = train["aglomerado"].isin(BAJA)
                resto = test[va.mascara_evaluacion(test) & ~test["aglomerado"].isin(BAJA)]
                y = resto[va.columna_respuesta(objetivo)].astype(float)
                for variante, sub in (("completo", train), ("sin_baja_respuesta", train[~baja_train])):
                    modelo = va.ajustar_linea_base(sub, objetivo, "decil_ratio")
                    cortes = modelo["decil"]["cortes"][1:-1]
                    filas_cortes.append({"objetivo_modelado": objetivo, "h": h, "anio_test": anio, "variante": variante,
                                         **{f"corte_{i + 1}": float(c) for i, c in enumerate(cortes)}})
                    filas_decil.append({"objetivo_modelado": objetivo, "h": h, "anio_test": anio, "variante": variante,
                                        "n_train": len(sub), **metricas_punto(objetivo, y, va.predecir_linea_base(modelo, resto),
                                                                              modelo["media_train"])})
cortes_decil = pd.DataFrame(filas_cortes)
decil_sin_baja = pd.DataFrame(filas_decil)
guardar(cortes_decil, "deciles_cortes")
guardar(decil_sin_baja, "deciles_metricas_resto")
if not decil_sin_baja.empty:
    clave = ["objetivo_modelado", "h", "anio_test"]
    metricas_decil = [c for c in ("rmse", "mae", "log_loss", "auc_roc", "brier") if c in decil_sin_baja]
    ancho = decil_sin_baja.pivot_table(index=clave, columns="variante", values=metricas_decil)
    for metrica in metricas_decil:
        ancho[(metrica, "sin_menos_completo")] = ancho[(metrica, "sin_baja_respuesta")] - ancho[(metrica, "completo")]
    display(ancho.xs("sin_menos_completo", axis=1, level=1).groupby(level=[0, 1]).agg(["median", "min", "max"]).round(5))
    display(cortes_decil[cortes_decil["anio_test"] == cortes_decil["anio_test"].max()].round(3))
medir("deciles", t0)

### Contexto: pobreza y Gini de los aglomerados de baja respuesta contra comparables

Comparables, según la regla fijada en `decisiones.md`: los 3 aglomerados con tasa ≥ 0,70 más cercanos en hogares
expandidos medios por trimestre, con `PONDERA` del hogar en la era regular 2016T2–2022T4. Se compara por año la
media de `tasa_pobreza_aglo_t` y `gini_ipcf_aglo_t` (año móvil en `t`) de los aglomerados-trimestre de train.

In [ ]:
partes = []
for anio, trimestre in [t for t in TRIMESTRES_TRAIN if era_de(*t) == "regional_2016"]:
    ruta = ARMONIZADO / "hogar" / f"ANO4={anio}/TRIMESTRE={trimestre}/data.parquet"
    tabla = pd.read_parquet(ruta, columns=["AGLOMERADO", "PONDERA"])
    partes.append(tabla.assign(aglomerado=tabla["AGLOMERADO"].astype(int),
                               PONDERA=pd.to_numeric(tabla["PONDERA"], errors="coerce"))
                  .groupby("aglomerado")["PONDERA"].sum().rename(f"{anio}T{trimestre}"))
tamano = pd.concat(partes, axis=1).mean(axis=1).rename("hogares_expandidos_medios")
alta = tabla_respuesta.loc[~tabla_respuesta["baja_respuesta"], "aglomerado"].astype(int).tolist()
comparables = {}
for a in CARACTERIZAR:
    candidatos = (tamano.loc[[c for c in alta if c in tamano.index and c != a]] - tamano.get(a, np.nan)).abs()
    comparables[a] = candidatos.nsmallest(N_COMPARABLES).index.astype(int).tolist()
tabla_comparables = pd.DataFrame([{"aglomerado": a, "nombre_codebook": nm.NOMBRES_AGLOMERADO[a],
                                   "hogares_expandidos_medios": tamano.get(a), "comparables": c,
                                   "nombres_comparables": [nm.NOMBRES_AGLOMERADO[x] for x in c]}
                                  for a, c in comparables.items()])
guardar(tabla_comparables, "comparables")
display(tabla_comparables)

contexto_t = (pares[1][va.mascara_train(pares[1])]
              .drop_duplicates(["aglomerado", "ANO4_t", "TRIMESTRE_t"])
              .groupby(["aglomerado", "ANO4_t"])[["tasa_pobreza_aglo_t", "gini_ipcf_aglo_t"]].mean().reset_index())
filas = []
for a, lista in comparables.items():
    propio = contexto_t[contexto_t["aglomerado"] == a].set_index("ANO4_t")
    pares_comp = contexto_t[contexto_t["aglomerado"].isin(lista)].groupby("ANO4_t")[["tasa_pobreza_aglo_t", "gini_ipcf_aglo_t"]].mean()
    union = propio[["tasa_pobreza_aglo_t", "gini_ipcf_aglo_t"]].join(pares_comp, rsuffix="_comparables", how="outer")
    filas.append(union.reset_index().assign(aglomerado=a, nombre_codebook=nm.NOMBRES_AGLOMERADO[a]))
contexto_baja = pd.concat(filas, ignore_index=True)
for variable in ("tasa_pobreza_aglo_t", "gini_ipcf_aglo_t"):
    contexto_baja[f"{variable}_diferencia"] = contexto_baja[variable] - contexto_baja[f"{variable}_comparables"]
guardar(contexto_baja, "contexto_baja_respuesta")
display(contexto_baja.groupby(["aglomerado", "nombre_codebook"])[
    ["tasa_pobreza_aglo_t", "tasa_pobreza_aglo_t_comparables", "tasa_pobreza_aglo_t_diferencia",
     "gini_ipcf_aglo_t", "gini_ipcf_aglo_t_comparables", "gini_ipcf_aglo_t_diferencia"]].mean().round(4))

### (a) Contaminación: `excluye` entrenado sin los aglomerados de baja respuesta

Se compara contra el modelo principal, los dos evaluados sobre el resto (test limpio sin esos aglomerados), con
diferencia pareada por bootstrap por vivienda. Si la diferencia es ~0, los aglomerados de baja respuesta no contaminan
el modelo para los demás.

In [ ]:
t0 = time.time()
tareas = []
if BAJA:
    for (objetivo, h), grupos in CONFIGURACIONES.items():
        lista = grupos["referencia"] + grupos["contaminacion"]
        for anio in anios[h]:
            tareas.append(({"pares": pares[h], "raiz": PREDICCIONES, "configuraciones": lista, "anio_test": int(anio),
                            "excluir_aglomerados": BAJA}, N_BOOT))
contaminacion = evaluar_tareas(tareas)
guardar(contaminacion, "contaminacion_resto")
if not contaminacion.empty:
    dif = contaminacion[contaminacion["configuracion"].str.endswith("_menos_excluye")
                        & contaminacion["configuracion"].str.startswith("gradient_boosting")]
    display(resumir(dif).pivot_table(index=["objetivo_modelado", "h", "metrica"],
                                     values=["mediana", "minimo", "maximo", "ic_mayor_0", "ic_menor_0", "n_folds"]).round(5))
medir("contaminacion", t0)

### (b) Desempeño del modelo principal por aglomerado contra su tasa de respuesta

Se juntan las predicciones limpias de todos los folds del GB `excluye` y del decil `excluye`. Por aglomerado:

- trayectoria: RMSE y skill contra el decil;
- entrada y permanencia: log loss, AUC y media predicha − tasa.

Al final, Spearman de cada métrica contra la tasa de declaración.

In [ ]:
filas = []
for (objetivo, h), grupos in CONFIGURACIONES.items():
    decil, gb = grupos["referencia"]
    unido = predicciones_limpias(gb, h).merge(predicciones_limpias(decil, h)[[*CLAVE_PAR, "fold", "prediccion"]],
                                               on=[*CLAVE_PAR, "fold"], suffixes=("", "_decil"))
    unido = unido.merge(pares[h][[*CLAVE_PAR, "aglomerado"]], on=CLAVE_PAR, how="left", validate="many_to_one")
    for aglomerado, g in unido.groupby("aglomerado"):
        y, p, base = g["y_observado"].astype(float), g["prediccion"].astype(float), g["prediccion_decil"].astype(float)
        if va.es_clasificacion(objetivo):
            resultado = va.metricas_clasificacion(y, p, base)
        else:
            resultado = va.metricas_regresion(y, p, float(y.mean()), base)
        filas.append({"objetivo_modelado": objetivo, "h": h, "aglomerado": int(aglomerado), **resultado})
por_aglomerado = pd.DataFrame(filas).merge(tabla_respuesta[["aglomerado", "nombre_codebook", "tasa_declaracion",
                                                            "baja_respuesta"]], on="aglomerado", how="left")
guardar(por_aglomerado, "desempeno_aglomerado")
filas = []
for (objetivo, h), g in por_aglomerado.groupby(["objetivo_modelado", "h"]):
    lista = ["rmse", "skill_vs_base"] if objetivo == "trayectoria" else ["log_loss", "auc_roc", "diferencia_media_predicha"]
    for metrica in lista:
        rho, valor_p = spearmanr(g["tasa_declaracion"], g[metrica], nan_policy="omit")
        filas.append({"objetivo_modelado": objetivo, "h": h, "metrica": metrica, "spearman_vs_tasa_declaracion": rho,
                      "p_valor": valor_p, "n_aglomerados": len(g),
                      "media_baja_respuesta": g.loc[g["baja_respuesta"], metrica].mean(),
                      "media_resto": g.loc[~g["baja_respuesta"], metrica].mean()})
correlacion_aglomerado = pd.DataFrame(filas)
guardar(correlacion_aglomerado, "desempeno_aglomerado_spearman")
display(correlacion_aglomerado.round(4))

### (c) Nivel de propensión: pesos por aglomerado y desempeño en los de baja respuesta

- Pesos de train del nivel `imputacion_excluye_reponderado_propension` por fold y aglomerado: mediana, p99 y Kish
  relativo.
- Desempeño del nivel contra `excluye`, solo en los aglomerados de baja respuesta, con diferencia pareada por
  bootstrap en las lecturas sin ponderar y ponderada por propensión.

In [ ]:
t0 = time.time()
filas = []
for h in HORIZONTES:
    clave_aglomerado = pares[h].set_index(CLAVE_PAR)["aglomerado"]
    for anio, peso in pesos_train[h].items():
        tabla = pd.DataFrame({"peso": peso.to_numpy(), "aglomerado": clave_aglomerado.reindex(peso.index).to_numpy()})
        tabla = tabla.dropna(subset=["aglomerado"])
        for aglomerado, g in tabla.groupby("aglomerado"):
            w = g["peso"].astype(float)
            filas.append({"h": h, "anio_test": anio, "aglomerado": int(aglomerado), "n": len(w), "peso_mediana": w.median(),
                          "peso_p99": w.quantile(0.99), "kish_relativo": float(w.sum() ** 2 / (w ** 2).sum() / len(w))})
pesos_aglomerado = pd.DataFrame(filas)
if not pesos_aglomerado.empty:
    pesos_aglomerado = pesos_aglomerado.merge(tabla_respuesta[["aglomerado", "nombre_codebook", "tasa_declaracion",
                                                                "baja_respuesta"]], on="aglomerado", how="left")
    guardar(pesos_aglomerado, "pesos_propension_aglomerado")
    ultimo = pesos_aglomerado[pesos_aglomerado["anio_test"] == pesos_aglomerado.groupby("h")["anio_test"].transform("max")]
    display(ultimo.sort_values(["h", "tasa_declaracion"]).round(3))

tareas = []
if BAJA:
    for (objetivo, h), grupos in CONFIGURACIONES.items():
        propension = [c for c in grupos["niveles"] if c.criterio_imputacion == PROPENSION]
        lista = grupos["referencia"] + propension
        for anio in anios[h]:
            for ponderacion in ("sin_ponderar", cte.PONDERACION_PROPENSION):
                peso = pesos_test[h][anio] if ponderacion == cte.PONDERACION_PROPENSION else None
                tareas.append(({"pares": pares[h], "raiz": PREDICCIONES, "configuraciones": lista, "anio_test": int(anio),
                                "ponderacion": ponderacion, "aglomerados": BAJA, "peso_test": peso}, N_BOOT))
propension_baja = evaluar_tareas(tareas)
guardar(propension_baja, "propension_baja_respuesta")
if not propension_baja.empty:
    dif = propension_baja[propension_baja["configuracion"].str.endswith("_menos_excluye")
                          & propension_baja["configuracion"].str.startswith("gradient_boosting")]
    display(resumir(dif).pivot_table(index=["objetivo_modelado", "h", "metrica"], columns="lectura",
                                     values=["mediana", "ic_mayor_0", "ic_menor_0"]).round(5))
medir("propension_aglomerado", t0)

## Caracterización de la baja respuesta

Hipótesis, no conclusiones [inferido].

**(i)** Tasa de declaración por año y por era de los aglomerados caracterizados (en la histórica, proporción de
hogares sin marca en `IDIMPH`), contra la mediana de los aglomerados de respuesta alta. Sirve para ver si la baja
respuesta es estable o se concentra en un período.

In [ ]:
tasa_anio = hogares.groupby(["aglomerado", "era", "ANO4"])["declara"].mean().rename("tasa_declaracion").reset_index()
alta_anio = (tasa_anio[tasa_anio["aglomerado"].isin(alta)].groupby(["era", "ANO4"])["tasa_declaracion"].median()
             .rename("mediana_respuesta_alta").reset_index())
tasa_anio = tasa_anio[tasa_anio["aglomerado"].isin(CARACTERIZAR)].merge(alta_anio, on=["era", "ANO4"], how="left")
tasa_anio["nombre_codebook"] = tasa_anio["aglomerado"].map(nm.NOMBRES_AGLOMERADO)
guardar(tasa_anio, "caracterizacion_tasa_anio")
display(tasa_anio.pivot_table(index="ANO4", columns="aglomerado", values="tasa_declaracion")
        .join(alta_anio.groupby("ANO4")["mediana_respuesta_alta"].mean()).round(3).T)

**(ii)** Propensión de respuesta por hogar, por era, fuera de fold (5 grupos por vivienda, como en el 05f §2), y su
AUC **dentro** de cada aglomerado. Además, la diferencia estandarizada declarantes − no declarantes en las X del
modelo de propensión (numéricas y dummies de cobertura, rama y región), con el desvío del aglomerado.

Se usan las X de propensión y no las del bloque A, porque las del bloque A derivan del ingreso y no existen para los
no declarantes.

Si en los aglomerados de baja respuesta la propensión discrimina menos y declarantes y no declarantes se parecen
más, es un indicio de no respuesta menos ligada al perfil del hogar. Va como hipótesis.

In [ ]:
t0 = time.time()
filas_auc, filas_smd = [], []
for era, grupo in hogares.groupby("era"):
    datos = nm.a_categoricas(grupo.reset_index(drop=True))
    fold = nm.grupos_vivienda(datos)
    datos["p_declara"], _ = nm.predecir_fuera_de_fold(datos[COLUMNAS_PROPENSION], datos["declara"], fold,
                                                      num_threads=mo.HILOS)
    numericas = [c for c in COLUMNAS_PROPENSION if not isinstance(datos[c].dtype, pd.CategoricalDtype)]
    dummies = pd.get_dummies(datos[["cobertura_jefe", "rama_jefe", "REGION"]].astype("string"), dummy_na=False).astype(float)
    matriz = pd.concat([datos[numericas].astype(float), dummies], axis=1)
    for aglomerado, g in datos.groupby("aglomerado"):
        metricas_aglo = nm.metricas_binarias(g["declara"], g["p_declara"])
        filas_auc.append({"era": era, "aglomerado": int(aglomerado), "n": metricas_aglo["n"], "auc": metricas_aglo["auc"],
                          "tasa_declaracion": metricas_aglo["tasa_observada"]})
        m = matriz.loc[g.index]
        declarante = g["declara"] == 1
        if declarante.all() or not declarante.any():
            continue
        desvio = m.std().replace(0, np.nan)
        smd = (m[declarante].mean() - m[~declarante].mean()) / desvio
        filas_smd.append({"era": era, "aglomerado": int(aglomerado), "smd_abs_medio": float(smd.abs().mean()),
                          "n_smd_mayor_0_1": int((smd.abs() > 0.1).sum()), "n_variables": int(smd.notna().sum())})
caracterizacion = pd.DataFrame(filas_auc).merge(pd.DataFrame(filas_smd), on=["era", "aglomerado"], how="left")
caracterizacion["grupo"] = np.select([caracterizacion["aglomerado"].isin(BAJA), caracterizacion["aglomerado"].isin(CARACTERIZAR)],
                                     ["baja_respuesta", "caracterizado"], default="respuesta_alta")
caracterizacion["nombre_codebook"] = caracterizacion["aglomerado"].map(nm.NOMBRES_AGLOMERADO)
guardar(caracterizacion, "caracterizacion_propension")
display(caracterizacion[caracterizacion["aglomerado"].isin(CARACTERIZAR)].sort_values(["aglomerado", "era"]).round(3))
display(caracterizacion.groupby(["era", "grupo"])[["auc", "smd_abs_medio", "n_smd_mayor_0_1"]].median().round(3))
medir("caracterizacion", t0)

## Tiempos, candado y registro de la corrida

In [ ]:
tiempos = pd.DataFrame([{"etapa": k, "segundos": v} for k, v in TIEMPOS.items()]).assign(
    folds=json.dumps({str(h): anios[h] for h in HORIZONTES}))
guardar(tiempos, "tiempos")
display(tiempos.round(0))

corridas = pd.read_csv(META / "corridas.csv", dtype=str)
aperturas = sum(json.loads(p).get("evento") == va.EVENTO_APERTURA for p in corridas["parametros_json"].dropna())
print(f"eventos {va.EVENTO_APERTURA} en corridas.csv: {aperturas}")
assert aperturas == 0
pm.registrar_corrida(META / "corridas.csv", NOTEBOOK,
                     {"REHACER": REHACER, "N_BOOT": N_BOOT, "folds": {str(h): anios[h] for h in HORIZONTES},
                      "criterios": mo.CRITERIOS_BLOQUE_C, "umbral_baja_respuesta": UMBRAL_BAJA_RESPUESTA,
                      "aglomerados_baja_respuesta": list(BAJA), "aglomerados_caracterizados": list(CARACTERIZAR),
                      "parametros_peso_propension": dict(nm.parametros_peso_propension(mo.HILOS)),
                      "hiperparametros_gb": {f"{k[0]}|{k[1]}": v for k, v in mo.HIPERPARAMETROS_GB.items()},
                      "version_x": mo.VERSION_X, "hilos": mo.HILOS, "folds_con_error": len(errores)},
                     int(sum(len(a) for a in anios.values())), escritas, time.time() - t0_proceso, COMMIT_GIT)
print(f"listo en {time.time() - t0_proceso:.0f}s")